# DS603 – Data Structures and Algorithms in Python
## Laboratory 08: Hierarchical Pointer Structures and Priority Queues


## Problem 01 — Linear Pointer Chains & Floyd’s Cycle Detection Engine

### Design
- Each `ListNode` stores an integer and a reference to the next node. The list maintains head, tail, and size so `append` is O(1).
- Reversal saves `current.next` before rewiring it, preventing the remainder of the chain from being disconnected. Reversal is O(N) time and O(1) auxiliary space.
- Cycle creation connects the tail to a zero-based position; `-1` means no cycle. Floyd’s algorithm uses slow/fast pointers and O(1) auxiliary space. After a collision, resetting one pointer to the head and advancing both one step finds the cycle entry.
- Printing stops after 100 nodes when cyclic, as specified. Reversal is rejected for cyclic lists because ordinary in-place reversal assumes a terminating chain.
- Complexity: append O(1); `to_list` O(min(N, 100)) for cyclic lists; reverse O(N); cycle detection O(N) time, O(1) space.

In [1]:
from __future__ import annotations
from typing import Optional


class ListNode:
    def __init__(self, val: int, next: Optional["ListNode"] = None) -> None:
        self.val = val
        self.next = next


class SinglyLinkedList:
    def __init__(self) -> None:
        self.head: Optional[ListNode] = None
        self.tail: Optional[ListNode] = None
        self.size = 0

    def append(self, val: int) -> None:
        node = ListNode(val)
        if self.head is None:
            self.head = self.tail = node
        else:
            assert self.tail is not None
            self.tail.next = node
            self.tail = node
        self.size += 1

    def to_list(self) -> list[int]:
        values: list[int] = []
        cycle_entry = self._cycle_entry_node()
        current = self.head
        if cycle_entry is not None:
            limit = min(self.size, 100)
            for _ in range(limit):
                if current is None:
                    break
                values.append(current.val)
                current = current.next
            if self.size > 100 or current is not None:
                values.append("...")  # type: ignore[arg-type]
            return values
        while current is not None:
            values.append(current.val)
            current = current.next
        return values
    def _cycle_entry_node(self) -> Optional[ListNode]:
        slow = fast = self.head
        while fast is not None and fast.next is not None:
            slow = slow.next  # type: ignore[assignment]
            fast = fast.next.next
            if slow is fast:
                slow = self.head
                while slow is not fast:
                    slow = slow.next  # type: ignore[assignment]
                    fast = fast.next  # type: ignore[assignment]
                return slow
        return None

    def detect_cycle_entry(self) -> Optional[int]:
        entry = self._cycle_entry_node()
        return entry.val if entry is not None else None

    def create_cycle(self, pos: int) -> None:
        if pos == -1:
            return
        if pos < -1 or pos >= self.size:
            raise ValueError("Cycle position is outside the list")
        current = self.head
        target = None
        index = 0
        while current is not None:
            if index == pos:
                target = current
                break
            current = current.next
            index += 1
        if self.tail is not None and target is not None:
            self.tail.next = target

    def reverse(self) -> bool:
        if self._cycle_entry_node() is not None:
            return False
        previous = None
        current = self.head
        self.tail = self.head
        while current is not None:
            next_temp = current.next
            current.next = previous
            previous = current
            current = next_temp
        self.head = previous
        return True

### Verification
The tests cover an empty list, single-node reversal, ordinary reversal, cycle entry detection, and refusing to reverse a cyclic list.

In [2]:
empty = SinglyLinkedList()
assert empty.to_list() == []
assert empty.detect_cycle_entry() is None
assert empty.reverse() is True

single = SinglyLinkedList()
single.append(9)
assert single.reverse() is True
assert single.to_list() == [9]

linked = SinglyLinkedList()
for value in [3, 2, 0, -4]:
    linked.append(value)
assert linked.to_list() == [3, 2, 0, -4]
assert linked.reverse() is True
assert linked.to_list() == [-4, 0, 2, 3]

cyclic = SinglyLinkedList()
for value in [3, 2, 0, -4]:
    cyclic.append(value)
cyclic.create_cycle(1)
assert cyclic.detect_cycle_entry() == 2
assert cyclic.reverse() is False
assert cyclic.to_list() == [3, 2, 0, -4, "..."]
print("Problem 01: all verification tests passed.")

Problem 01: all verification tests passed.


### Optional command-line driver
Input/output follows the operation format in the handout. Run this cell only when you want to test interactive input.

In [3]:
def run_problem_01() -> None:
    q = int(input().strip())
    linked = SinglyLinkedList()
    for _ in range(q):
        parts = input().split()
        command = parts[0].upper()
        if command == "APPEND":
            value = int(parts[1])
            linked.append(value)
            print(f"Appended: {value}")
        elif command == "PRINT":
            values = linked.to_list()
            print(f"List: {values}")
        elif command == "REVERSE":
            if linked.reverse():
                print("List Reversed")
            else:
                print("Cannot reverse cyclic list")
        elif command == "CYCLE":
            pos = int(parts[1])
            linked.create_cycle(pos)
            print(f"Cycle created at pos {pos}")
        elif command == "DETECT":
            entry = linked.detect_cycle_entry()
            if entry is None:
                print("No Cycle Detected")
            else:
                print(f"Cycle Detected at Node: {entry}")

## Problem 02 — Dynamic Binary Search Tree (BST) Engine & Inorder Deletion

### Design
- The invariant is that every key in a node’s left subtree is smaller and every key in its right subtree is larger.
- Insertion and search follow one root-to-leaf path; duplicate keys are rejected with `Duplicate Key: <key>`.
- Deletion handles three cases: leaf (return `None`), one child (return that child), and two children (copy the minimum key in the right subtree, then delete that successor).
- BST validation propagates strict `(low, high)` bounds; local parent-child checks alone are insufficient.
- Complexity: insert/search/delete O(H), inorder O(N), validation O(N); H can be N in an unbalanced tree, so operations can be O(N) worst case. Recursive methods use O(H) call-stack space.

In [4]:
class TreeNode:
    def __init__(
        self, val: int, left: Optional["TreeNode"] = None,
        right: Optional["TreeNode"] = None
    ) -> None:
        self.val = val
        self.left = left
        self.right = right


class BinarySearchTree:
    def __init__(self) -> None:
        self.root: Optional[TreeNode] = None

    def insert(self, key: int) -> bool:
        if self.root is None:
            self.root = TreeNode(key)
            return True
        current = self.root
        while True:
            if key == current.val:
                return False
            if key < current.val:
                if current.left is None:
                    current.left = TreeNode(key)
                    return True
                current = current.left
            else:
                if current.right is None:
                    current.right = TreeNode(key)
                    return True
                current = current.right

    def search(self, key: int) -> bool:
        current = self.root
        while current is not None:
            if key == current.val:
                return True
            current = current.left if key < current.val else current.right
        return False

    def _delete(self, node: Optional[TreeNode], key: int) -> Optional[TreeNode]:
        if node is None:
            return None
        if key < node.val:
            node.left = self._delete(node.left, key)
        elif key > node.val:
            node.right = self._delete(node.right, key)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            successor = node.right
            while successor.left is not None:
                successor = successor.left
            node.val = successor.val
            node.right = self._delete(node.right, successor.val)
        return node

    def delete(self, key: int) -> None:
        self.root = self._delete(self.root, key)

    def inorder(self) -> list[int]:
        result: list[int] = []
        def visit(node: Optional[TreeNode]) -> None:
            if node is None:
                return
            visit(node.left)
            result.append(node.val)
            visit(node.right)
        visit(self.root)
        return result

    def is_valid(self) -> bool:
        def validate(node: Optional[TreeNode], low: float, high: float) -> bool:
            if node is None:
                return True
            if not low < node.val < high:
                return False
            return (validate(node.left, low, node.val)
                    and validate(node.right, node.val, high))
        return validate(self.root, float("-inf"), float("inf"))

### Verification
Tests include an empty tree, duplicate rejection, search hits/misses, deletion of a leaf, a one-child node, a two-child node, and a deliberately invalid BST.

In [5]:
bst = BinarySearchTree()
assert bst.inorder() == []
assert bst.is_valid() is True
assert bst.insert(50)
for value in [30, 70, 20, 40, 60, 80]:
    assert bst.insert(value)
assert not bst.insert(50)  # duplicate rejected
assert bst.search(40) and not bst.search(999)
assert bst.is_valid()
bst.delete(20)  # leaf
assert bst.inorder() == [30, 40, 50, 60, 70, 80]
bst.insert(35)
bst.delete(40)  # node with one child (35)
assert bst.inorder() == [30, 35, 50, 60, 70, 80]
bst.delete(50)  # two children; successor is 60
assert bst.inorder() == [30, 35, 60, 70, 80]
assert bst.is_valid()

invalid = TreeNode(10, TreeNode(5, None, TreeNode(12)), TreeNode(20))
assert BinarySearchTree().is_valid()
def validate_arbitrary_root(root: Optional[TreeNode]) -> bool:
    def check(node: Optional[TreeNode], low: float, high: float) -> bool:
        if node is None:
            return True
        if not low < node.val < high:
            return False
        return check(node.left, low, node.val) and check(node.right, node.val, high)
    return check(root, float("-inf"), float("inf"))
assert not validate_arbitrary_root(invalid)
print("Problem 02: all verification tests passed.")

Problem 02: all verification tests passed.


### Optional command-line driver

In [6]:
def run_problem_02() -> None:
    q = int(input().strip())
    bst = BinarySearchTree()
    for _ in range(q):
        parts = input().split()
        command, key_text = parts[0].upper(), parts[1] if len(parts) > 1 else None
        if command == "INSERT":
            key = int(key_text)
            if bst.insert(key):
                print(f"Inserted: {key}")
            else:
                print(f"Duplicate Key: {key}")
        elif command == "SEARCH":
            print(f"Found: {bst.search(int(key_text))}")
        elif command == "DELETE":
            key = int(key_text)
            bst.delete(key)
            print(f"Deleted: {key}")
        elif command == "INORDER":
            print(f"Inorder: {bst.inorder()}")
        elif command == "VALID":
            print(f"Valid BST: {bst.is_valid()}")

## Problem 03 — Queue-Based Level-Order Traversal & Lowest Common Ancestor

### Design
- Level-order traversal uses a FIFO `deque`. At the beginning of each level, snapshot `len(queue)` and process exactly that many nodes so children are grouped into the next level.
- `lca_bst` uses the BST ordering property: move left if both keys are smaller, right if both are larger, otherwise the current node is the split point.
- `lca_binary_tree` uses postorder recursion: return the current node if it matches either target; if both subtrees return a result, the current node is the LCA.
- Complexity: level-order O(N) time and O(W) queue space where W is maximum width; BST LCA O(H); general-tree LCA O(N) time and O(H) recursion space.
- The parser below builds a tree from a comma-separated level-order representation with `null` placeholders.

In [7]:
from collections import deque


def level_order(root: Optional[TreeNode]) -> list[list[int]]:
    if root is None:
        return []
    result: list[list[int]] = []
    queue = deque([root])
    while queue:
        level_size = len(queue)
        level: list[int] = []
        for _ in range(level_size):
            node = queue.popleft()
            level.append(node.val)
            if node.left is not None:
                queue.append(node.left)
            if node.right is not None:
                queue.append(node.right)
        result.append(level)
    return result


def lca_bst(root: Optional[TreeNode], p: int, q: int) -> Optional[int]:
    current = root
    while current is not None:
        if p < current.val and q < current.val:
            current = current.left
        elif p > current.val and q > current.val:
            current = current.right
        else:
            return current.val
    return None


def lca_binary_tree(root: Optional[TreeNode], p: int, q: int) -> Optional[int]:
    def helper(node: Optional[TreeNode]) -> Optional[TreeNode]:
        if node is None:
            return None
        if node.val == p or node.val == q:
            return node
        left_result = helper(node.left)
        right_result = helper(node.right)
        if left_result is not None and right_result is not None:
            return node
        return left_result if left_result is not None else right_result

    answer = helper(root)
    return answer.val if answer is not None else None


def build_tree_level_order(serialized: str) -> Optional[TreeNode]:
    tokens = [token.strip() for token in serialized.split(",")]
    if not tokens or tokens[0].lower() == "null" or tokens[0] == "":
        return None
    root = TreeNode(int(tokens[0]))
    queue = deque([root])
    index = 1
    while queue and index < len(tokens):
        node = queue.popleft()
        if index < len(tokens) and tokens[index].lower() != "null":
            node.left = TreeNode(int(tokens[index]))
            queue.append(node.left)
        index += 1
        if index < len(tokens) and tokens[index].lower() != "null":
            node.right = TreeNode(int(tokens[index]))
            queue.append(node.right)
        index += 1
    return root

### Verification
The test tree matches the handout’s example: `3,5,1,6,2,0,8,null,null,7,4`.

In [8]:
tree = build_tree_level_order("3,5,1,6,2,0,8,null,null,7,4")
assert level_order(tree) == [[3], [5, 1], [6, 2, 0, 8], [7, 4]]
assert lca_binary_tree(tree, 5, 1) == 3
assert lca_binary_tree(tree, 5, 4) == 5
assert lca_binary_tree(tree, 7, 4) == 2
assert lca_binary_tree(tree, 100, 4) == 4  # algorithm returns the present target
assert level_order(None) == []

bst_lca_tree = build_tree_level_order("6,2,8,0,4,7,9,null,null,3,5")
assert lca_bst(bst_lca_tree, 2, 8) == 6
assert lca_bst(bst_lca_tree, 2, 4) == 2
assert lca_bst(bst_lca_tree, 100, 101) is None
print("Problem 03: all verification tests passed.")

Problem 03: all verification tests passed.


### Optional command-line driver
The handout names `LEVELS` and `LCA_BT p q` operations. This driver also supports `LCA_BST p q` when the supplied tree is a BST.

In [9]:
def run_problem_03() -> None:
    serialized = input().strip()
    root = build_tree_level_order(serialized)
    q = int(input().strip())
    for _ in range(q):
        parts = input().split()
        command = parts[0].upper()
        if command == "LEVELS":
            print(f"Levels: {level_order(root)}")
        elif command in {"LCA_BT", "LCA_BST"}:
            p, q_value = int(parts[1]), int(parts[2])
            if command == "LCA_BT":
                answer = lca_binary_tree(root, p, q_value)
            else:
                answer = lca_bst(root, p, q_value)
            print(f"LCA({p}, {q_value}) = {answer}")

## Problem 04 — Industrial Array-Based MinHeap Engine & O(N) Heapify

### Design
- The heap is stored in a plain list. For index `i`, parent is `(i - 1) // 2`, left child is `2*i + 1`, and right child is `2*i + 2`.
- `push` appends and sifts up in O(log N). `pop_min` swaps the root with the last element, removes the last element, and sifts down in O(log N). `peek` is O(1).
- `build_heap` uses Floyd’s bottom-up method: sift down from `N//2 - 1` to zero. Leaves need no work. This is O(N), unlike N repeated pushes which take O(N log N) in the worst case.
- `heapsort` repeatedly removes the minimum from a heap and therefore returns ascending order in O(N log N) time.
- No `heapq` or other heap library is used.

In [10]:
class MinHeap:
    def __init__(self) -> None:
        self.data: list[int] = []

    @staticmethod
    def parent(index: int) -> int:
        return (index - 1) // 2

    @staticmethod
    def left(index: int) -> int:
        return 2 * index + 1

    @staticmethod
    def right(index: int) -> int:
        return 2 * index + 2

    def _sift_up(self, index: int) -> None:
        while index > 0:
            parent_index = self.parent(index)
            if self.data[parent_index] <= self.data[index]:
                break
            self.data[parent_index], self.data[index] = (
                self.data[index], self.data[parent_index]
            )
            index = parent_index

    def _sift_down(self, index: int) -> None:
        size = len(self.data)
        while True:
            smallest = index
            left_index = self.left(index)
            right_index = self.right(index)
            if left_index < size and self.data[left_index] < self.data[smallest]:
                smallest = left_index
            if right_index < size and self.data[right_index] < self.data[smallest]:
                smallest = right_index
            if smallest == index:
                break
            self.data[index], self.data[smallest] = (
                self.data[smallest], self.data[index]
            )
            index = smallest

    def push(self, val: int) -> None:
        self.data.append(val)
        self._sift_up(len(self.data) - 1)

    def pop_min(self) -> int:
        if not self.data:
            raise IndexError("pop_min from empty heap")
        if len(self.data) == 1:
            return self.data.pop()
        minimum = self.data[0]
        self.data[0] = self.data.pop()
        self._sift_down(0)
        return minimum

    def peek(self) -> int:
        if not self.data:
            raise IndexError("peek from empty heap")
        return self.data[0]

    def build_heap(self, arr: list[int]) -> None:
        self.data = list(arr)
        for index in range(len(self.data) // 2 - 1, -1, -1):
            self._sift_down(index)

    @staticmethod
    def heapsort(arr: list[int]) -> list[int]:
        heap = MinHeap()
        heap.build_heap(arr)
        result: list[int] = []
        while heap.data:
            result.append(heap.pop_min())
        return result

### Verification
Checks include empty-heap errors, one element, duplicate values, negative values, heap order after building/pushing/popping, and ascending heapsort.

In [11]:
heap = MinHeap()
for method in (heap.peek, heap.pop_min):
    try:
        method()
        raise AssertionError("Expected IndexError for an empty heap")
    except IndexError:
        pass

heap.build_heap([9, 4, 7, 1, -2, 6, 5])
assert heap.peek() == -2
assert heap.data == [-2, 1, 5, 9, 4, 6, 7]
assert heap.pop_min() == -2
assert heap.peek() == 1
heap.push(-10)
assert heap.peek() == -10
assert MinHeap.heapsort([12, 3, 17, 8, 34, 5]) == [3, 5, 8, 12, 17, 34]
assert MinHeap.heapsort([]) == []
assert MinHeap.heapsort([4]) == [4]
assert MinHeap.heapsort([2, 2, -1, 2]) == [-1, 2, 2, 2]
print("Problem 04: all verification tests passed.")

Problem 04: all verification tests passed.


### Optional command-line driver

In [12]:
def run_problem_04() -> None:
    q = int(input().strip())
    heap = MinHeap()
    for _ in range(q):
        parts = input().split()
        command = parts[0].upper()
        if command == "BUILD":
            values = [int(value) for value in parts[1:]]
            heap.build_heap(values)
            print(f"Heap Built (size {len(heap.data)})")
        elif command == "PUSH":
            value = int(parts[1])
            heap.push(value)
            print(f"Pushed: {value}")
        elif command == "POP":
            try:
                print(f"Popped Min: {heap.pop_min()}")
            except IndexError:
                print("Heap is empty")
        elif command == "PEEK":
            try:
                print(f"Min: {heap.peek()}")
            except IndexError:
                print("Heap is empty")
        elif command == "SORT":
            values = [int(value) for value in parts[1:]]
            print(f"Sorted: {MinHeap.heapsort(values)}")
        elif command == "DUMP":
            print(f"Array: {heap.data}")

## Problem 05 — Real-Time Streaming Median Analytics Engine (Capstone)

### Design
- `small` is a max-heap for the lower half, represented by negated integers in a manually implemented min-heap. `large` is a min-heap for the upper half.
- On insertion, move the maximum of `small` to `large`, then rebalance if `large` has more elements than `small`.
- Invariant: `len(small) == len(large)` or `len(small) == len(large) + 1`; all values in the lower half are no greater than those in the upper half.
- Odd count: median is the top of `small` after undoing negation. Even count: average the two heap tops.
- Complexity: insertion O(log N), median query O(1), storage O(N). Empty median queries raise `ValueError`.

In [13]:
class StreamingMedianTracker:
    def __init__(self) -> None:
        self.small = MinHeap()  # stores negative values: max-heap behavior
        self.large = MinHeap()  # ordinary min-heap

    def add_number(self, num: int) -> None:
        self.small.push(-num)
        largest_lower = -self.small.pop_min()
        self.large.push(largest_lower)

        if len(self.large.data) > len(self.small.data):
            smallest_upper = self.large.pop_min()
            self.small.push(-smallest_upper)

    def find_median(self) -> float:
        if not self.small.data:
            raise ValueError("Median is undefined for an empty stream")
        if len(self.small.data) > len(self.large.data):
            return float(-self.small.peek())
        return (-self.small.peek() + self.large.peek()) / 2.0

### Verification
The test cases check empty-stream behavior, odd/even counts, negative values, duplicates, and the sample sequence from the handout.

In [14]:
tracker = StreamingMedianTracker()
try:
    tracker.find_median()
    raise AssertionError("Expected ValueError on an empty tracker")
except ValueError:
    pass

tracker.add_number(5)
assert tracker.find_median() == 5.0
tracker.add_number(15)
assert tracker.find_median() == 10.0
tracker.add_number(1)
assert tracker.find_median() == 5.0
tracker.add_number(3)
assert tracker.find_median() == 4.0

negative_tracker = StreamingMedianTracker()
for value in [-5, -1, -3]:
    negative_tracker.add_number(value)
assert negative_tracker.find_median() == -3.0
negative_tracker.add_number(-3)
assert negative_tracker.find_median() == -3.0
assert len(negative_tracker.small.data) in {
    len(negative_tracker.large.data),
    len(negative_tracker.large.data) + 1
}
print("Problem 05: all verification tests passed.")

Problem 05: all verification tests passed.
